# Class 6 - Advanced Python

Today we learn to write our own **functions**, turn notebook workflows into **scripts**, solve a simple differential equation with the **Euler method**, and **measure** how fast our code runs.

Goals:
- define and call your own Python functions (`def`, parameters, `return`),
- refactor notebook workflows from Classes 4 and 5 into reusable scripts,
- solve a first-order ODE with the Euler method,
- compare loop vs vectorised performance with `time.perf_counter()`.

In [ ]:
import numpy as np

## 1. Creating functions

In **Class 3b**, you converted garden temperatures from Celsius to Fahrenheit for your American cousin. If you only had two values, copy-pasting the formula is fine:

```python
temp_f1 = temp_c1 * 9 / 5 + 32
temp_f2 = temp_c2 * 9 / 5 + 32
```

But repeating the same logic many times makes code long and hard to maintain. A **function** packages instructions into a reusable block. You **define** it once, then **call** it whenever you need it.

Recommended reading (we follow a similar path, but with °C → °F): [Software Carpentry — Creating Functions](https://swcarpentry.github.io/python-novice-inflammation/08-func.html) (stop at *Tidying up*).

### i. Repetitive code

Three garden readings in °C — convert each one by hand:

In [ ]:
TempC1 = 20.4
TempC2 = 20.8
TempC3 = 21.1

TempF1 = TempC1 * 9 / 5 + 32
TempF2 = TempC2 * 9 / 5 + 32
TempF3 = TempC3 * 9 / 5 + 32

print(TempF1, TempF2, TempF3)

### ii. Defining a function with `def` and `return`

A function definition starts with `def`, then the function name, parameters in parentheses, and a colon. The body is **indented**. Use `return` to send a result back to the caller.

In [ ]:
def celsius_to_fahrenheit(temp_c):
    return temp_c * 9 / 5 + 32

In [ ]:
celsius_to_fahrenheit(0)

In [ ]:
print('freezing point of water:', celsius_to_fahrenheit(0), '°F')
print('boiling point of water:', celsius_to_fahrenheit(100), '°F')

### iii. Composing functions

Larger programs are built from small functions combined together. Celsius → kelvin, then kelvin from Fahrenheit via composition:

In [ ]:
def celsius_to_kelvin(temp_c):
    return temp_c + 273.15

def fahrenheit_to_celsius(temp_f):
    return (temp_f - 32) * 5 / 9

def fahrenheit_to_kelvin(temp_f):
    temp_c = fahrenheit_to_celsius(temp_f)
    return celsius_to_kelvin(temp_c)

print('boiling point of water in kelvins:', fahrenheit_to_kelvin(212.0))

### iv. Variable scope

Variables created **inside** a function are **local**: they disappear when the function finishes. If you try to use them outside, Python raises a `NameError`.

In [ ]:
def demo_scope(temp_c):
    temp_k = celsius_to_kelvin(temp_c)
    return temp_k

temp_kelvins = demo_scope(20.0)
print('temperature in kelvins was:', temp_kelvins)

# Uncomment the next line to see the NameError:
# print(temp_k)

A variable defined **outside** any function is **global**. A function can read global variables (but should not modify them without good reason):

In [ ]:
def print_temperatures():
    print('temperature in Celsius was:', temp_celsius)
    print('temperature in kelvins was:', temp_kelvins)

temp_celsius = 20.0
temp_kelvins = celsius_to_kelvin(temp_celsius)
print_temperatures()

## Exercise A — American cousin with a function (Class 3b recap)

This exercise is a copy of **Class 3b Exercise A**, except that you must wrap the conversion in a function.

To find the best location to plant a temperature-sensitive species in your garden, you measure the temperature at five different locations. The numbers you read on your thermometer are: 20.4°C, 20.8°C, 21.1°C, 20.3°C, and 21.2°C.

Your American cousin, an avid gardener, comes to visit you and doesn't understand degrees Celsius. Convert all temperature measures to degrees Fahrenheit. Display the averages in both units.

Tasks:
1. Write `celsius_to_fahrenheit(temp_c)` for a single value.
2. Store the five readings in a NumPy array.
3. Use a `for` loop to fill a second array with Fahrenheit values (call your function inside the loop).
4. Print the Celsius and Fahrenheit averages.

In [ ]:
# Solution
def celsius_to_fahrenheit(temp_c):
    return temp_c * 9 / 5 + 32

TempGardenC = np.array([20.4, 20.8, 21.1, 20.3, 21.2])
NObs = np.size(TempGardenC)

TempGardenF = np.zeros(NObs)
for i in range(NObs):
    TempGardenF[i] = celsius_to_fahrenheit(TempGardenC[i])

print(f'Average (°C): {np.mean(TempGardenC):.2f}')
print(f'Average (°F): {np.mean(TempGardenF):.2f}')

## 2. Functions on real data (Classes 4 & 5 recap)

In **Class 4**, you wrote `add_fahrenheit_column(df, celsius_col)` to convert a Pandas column for your American cousin. In **Class 5**, you wrote `celsius_to_fahrenheit_da(temp_da)` for an xarray DataArray.

The conversion formula is always the same — only the **data container** changes (NumPy array, Pandas column, xarray DataArray). In this class you learned the general idea behind those functions: define once with `def`, call many times.

## 3. From notebook to script: Class 5 `post-processing.py`

In **Class 5**, you analysed gridded temperature data step by step in a notebook. For reproducibility, the same workflow can live in a **Python script** that anyone can run from the terminal:

```bash
python post-processing.py
```

Open [`post-processing.py`](post-processing.py) in VS Code. It contains the Class 5 pipeline:

1. Open netCDF and extract `pred_temperature_C`
2. Hartheim point series
3. Freiburg box → spatial mean → daily resample
4. Value mask (`Temp > 5`)
5. Urban heat island stats (**Exercise D** from Class 5)

Each step is already a small function (`load_temperature`, `freiburg_box`, `spatial_mean_series`, …). Run the script:

In [ ]:
%run post-processing.py

## Exercise D — Multiple netCDF files

In research you rarely have a single file. What changes if you receive **several monthly netCDF files** (same variable, same grid)?

Tasks:
1. Think: which parts of `post-processing.py` should become a reusable function?
2. Write `process_netcdf(path, city_mask_path)` that returns a summary dict with:
   - file path,
   - Hartheim mean temperature (°C),
   - daily spatial-mean series over the Freiburg box,
   - urban heat island stats (hours and % with city − countryside > 1 °C).
3. Loop over a list of paths and collect the results.

For practice, we split January into two halves (no extra data download needed). The instructor solution is in [`post-processing_solution.py`](post-processing_solution.py).

In [ ]:
# Setup: create two netCDF chunks from the January file (run once)
from pathlib import Path

ChunkDir = Path('_exercise_chunks')
ChunkDir.mkdir(exist_ok=True)

TempDs = xr.open_dataset('../../processed_data/pred_temperature_C_2024_01_wgs84.nc')
TempDs.sel(time=slice('2024-01-01', '2024-01-15')).to_netcdf(ChunkDir / 'jan_2024_part1.nc')
TempDs.sel(time=slice('2024-01-16', '2024-01-31')).to_netcdf(ChunkDir / 'jan_2024_part2.nc')
TempDs.close()
print('Created', list(ChunkDir.glob('*.nc')))

In [ ]:
# Solution — see post-processing_solution.py for the full version
%run post-processing_solution.py

## 4. Same pattern for Class 4: `station-processing.py`

The same idea applies to **tabular** data from **Class 4**. Open [`station-processing.py`](station-processing.py): it loads the Hartheim CSV, selects columns, adds a gradient column, counts hot hours, and resamples to daily values.

Run it:

In [ ]:
%run station-processing.py

## Exercise E — Multiple station CSVs

What if you monitor **several climate stations** (one CSV per site)?

Tasks:
1. Refactor the Class 4 workflow into `process_station_csv(csv_path, columns)`.
2. Return a summary dict: path, number of rows, mean 2 m temperature, number of hot hours (> 38 °C), daily DataFrame.
3. Loop over a list of CSV paths.

We use the real Hartheim file plus a synthetic second station (subset of dates, slightly warmer). The instructor solution is in [`station-processing_solution.py`](station-processing_solution.py).

In [ ]:
# Solution — see station-processing_solution.py for the full version
%run station-processing_solution.py

## 5. The Euler method (first-order ODE)

A **first-order ordinary differential equation (ODE)** has the form:

$$\frac{dy}{dt} = f(t, y)$$

We know the initial value $y(t_0) = y_0$ and want to predict $y$ at later times.

The **Euler method** takes small steps forward in time:

$$y_{n+1} = y_n + \Delta t \cdot f(t_n, y_n)$$

At each step, we use the **slope** of the function at the current point ($f(t_n, y_n)$) to estimate where $y$ will be after a time step $\Delta t$. Smaller $\Delta t$ usually gives better accuracy, but requires more steps.

### Newton cooling

After sunset, air near the ground cools toward a stable night-time temperature. A simple model is:

$$\frac{dT}{dt} = -k \,(T - T_{eq})$$

where $T_{eq}$ is the equilibrium (night-time) temperature and $k$ controls how fast cooling happens. This is a **first-order linear ODE** — a good candidate for Euler.

We will compare the Euler prediction with **observed** Hartheim `temperature_2m` from **Class 4** for one summer night.

## Exercise F — Euler method on Hartheim data

Tasks:
1. Write `euler_step(y, dt, f)` that returns $y_{n+1}$ for one step.
2. Write `euler_solve(y0, t_array, f)` that loops over `t_array` and returns the predicted values.
3. Define `cooling_rate(T, k, T_eq)` implementing $f = -k(T - T_{eq})$.
4. Load Hartheim `temperature_2m` for `2026-07-24 18:00` → `2026-07-25 06:00`.
5. Use the first observed hour as $T_0$, set $T_{eq}$ from the last observed value, pick $k = 0.3$ (1/h), $\Delta t = 1$ h.
6. Plot observed vs Euler-predicted temperatures.
7. What happens if you double $\Delta t$?

In [ ]:
# Solution
import matplotlib.pyplot as plt
import pandas as pd

HartheimRaw = pd.read_csv('../../processed_data/26030350_hourly.csv', parse_dates=['utc_time'])
HartheimData = HartheimRaw.set_index('utc_time')

def euler_step(y, dt, f):
    return y + dt * f(y)

def euler_solve(y0, t_array, f):
    y = np.zeros(len(t_array))
    y[0] = y0
    for n in range(len(t_array) - 1):
        dt = (t_array[n + 1] - t_array[n]).total_seconds() / 3600  # hours
        y[n + 1] = euler_step(y[n], dt, f)
    return y

def cooling_rate(T, k, T_eq):
    return -k * (T - T_eq)

# Load one night of Hartheim data
NightData = HartheimData.loc['2026-07-24 18:00':'2026-07-25 06:00', 'temperature_2m']
T0 = float(NightData.iloc[0])
T_eq = float(NightData.iloc[-1])
k = 0.3

f = lambda T: cooling_rate(T, k, T_eq)
TEuler = euler_solve(T0, NightData.index, f)

plt.plot(NightData.index, NightData.values, 'o-', label='Observed')
plt.plot(NightData.index, TEuler, 's--', label='Euler (dt = 1 h)')
plt.ylabel('Temperature (°C)')
plt.title('Newton cooling — Hartheim night 24–25 July 2026')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# What if we double dt? (use every other hour as time steps)
CoarseIndex = NightData.index[::2]
TEulerCoarse = euler_solve(T0, CoarseIndex, f)

plt.plot(NightData.index, NightData.values, 'o-', label='Observed')
plt.plot(CoarseIndex, TEulerCoarse, 's--', label='Euler (dt ≈ 2 h)')
plt.ylabel('Temperature (°C)')
plt.title('Larger time step → larger error')
plt.legend()
plt.grid(True)
plt.show()

## 6. Profiling — how fast is your code?

In **Class 3b** you chose between a `for` loop and **vectorisation**. With only five garden temperatures, both approaches feel instant. But with millions of values, the difference matters.

Python's `time.perf_counter()` returns the current time in seconds with high precision. Wrap code in a timer to measure elapsed time:

```python
import time
start = time.perf_counter()
# ... your code ...
elapsed = time.perf_counter() - start
print(f'Elapsed: {elapsed:.4f} s')
```

Open [`profiling.py`](profiling.py): it repeats the American-cousin conversion from Classes 3a/3b three ways (separate variables, loop, vectorisation), first on 5 values, then on 1 000 000 values.

In [ ]:
%run profiling.py

**Take-home message:** for large numerical arrays, prefer vectorised NumPy operations over Python `for` loops. Functions let you package each approach cleanly and compare them fairly with a timer.